# Netflix Movies and TV Shows — Data Cleaning

**Goal:** clean the Netflix titles dataset with Pandas, and make a deliberate decision for every column instead of just dropping nulls.

## Part 1 — Load and inspect

**1.1 Imports.** `Path` makes the file path work on any computer.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

**1.2 Load the raw data.** We keep an untouched copy called `raw` and do all cleaning on `df`. That lets us compare before and after, and if we break `df` we can restart from `raw` without reloading the file.

In [2]:
data_path = Path("raw_data/netflix_titles.csv")

raw = pd.read_csv(data_path)
df = raw.copy()

df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno ...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swi..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noure...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robb..."
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice C..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Ar...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies","In a city of coaching centers known to train India’s finest collegiate minds, an earnest but une..."


**1.3 Structure.** Look at the `Dtype` column. `date_added` and `duration` are text, which is part of the problem.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


**1.4 Summary of every column.** `include="all"` also covers the text columns, not only `release_year`.

In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8807,8807,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8807,2,Movie,6131,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8807,8807,Dick Johnson Is Dead,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,6173,4528,Rajiv Chilaka,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cast,7982,7692,David Attenborough,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,7976,748,United States,2818,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8797,1767,"January 1, 2020",109,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8807.0,NaN,NaN,NaN,2014.180198,8.819312,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8803,17,TV-MA,3207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8804,220,1 Season,1793,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Part 2 — Assess the data quality before changing anything

**2.1 Quality report.** One table with type, missing count, missing percent and unique values for every column.

In [5]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate show_id: {df['show_id'].duplicated().sum():,}")

quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(),
}).sort_values("missing_count", ascending=False)

quality_report

Rows: 8,807
Columns: 12
Duplicate rows: 0
Duplicate show_id: 0


,data_type,missing_count,missing_percent,unique_values
director,str,2634,29.91,4528
country,str,831,9.44,748
cast,str,825,9.37,7692
date_added,str,10,0.11,1767
rating,str,4,0.05,17
duration,str,3,0.03,220
show_id,str,0,0.00,8807
type,str,0,0.00,2
title,str,0,0.00,8807
release_year,int64,0,0.00,74


**Why missing values are not all the same.** `director`, `cast` and `country` are descriptive fields: many titles simply have no listed value, so deleting those rows would throw away thousands of valid titles. `date_added`, `rating` and `duration` are small counts but affect analysis, so they need a closer look.

**2.2 Look at the few missing `rating` rows.** Note the `duration` of each.

In [6]:
df.loc[df["rating"].isna(), ["show_id", "type", "title", "release_year", "duration"]]

,show_id,type,title,release_year,duration
5989,s5990,Movie,13TH: A Conversation with Oprah Winfrey & Ava DuVernay,2017,37 min
6827,s6828,TV Show,Gargantia on the Verdurous Planet,2013,1 Season
7312,s7313,TV Show,Little Lunch,2015,1 Season
7537,s7538,Movie,My Honor Was Loyalty,2015,115 min


**2.3 Look at the missing `duration` rows.** Now look at the `rating` of each. Do you notice something odd?

In [7]:
df.loc[df["duration"].isna(), ["show_id", "type", "title", "rating"]]

,show_id,type,title,rating
5541,s5542,Movie,Louis C.K. 2017,74 min
5794,s5795,Movie,Louis C.K.: Hilarious,84 min
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,66 min


## Part 3 — Clean, one decision at a time

| Column | Decision | Why |
|---|---|---|
| `rating` / `duration` | Move the 3 misplaced runtimes | Values in the wrong column |
| `duration` | Split into a number and a unit | Mixes `90 min` and `2 Seasons` |
| `director`, `cast`, `country` | Fill with `Unknown` | Keeps every title |
| `rating` | Fill the rest with `Not Rated` | Real rating can't be verified from this file |
| `date_added` | Convert to datetime | Text dates can't be sorted or filtered |

**3.1 Move the misplaced runtimes.** `fullmatch` selects only ratings that are *entirely* a number plus `min`. Order matters: this must run **before** the duration split, because the split needs these 3 runtimes to already be in `duration`.

In [18]:
# Ratings that are really movie runtimes
misplaced = df["rating"].str.fullmatch(r"\s*\d+\s*min\s*", na=False)

# Copy the runtime into the correct column
df.loc[misplaced, "duration"] = df.loc[misplaced, "rating"]

# The rating for these rows is now unknown
df.loc[misplaced, "rating"] = pd.NA

print("Rows moved:", misplaced.sum())
print("Missing rating:", df["rating"].isna().sum())
print("Missing duration:", df["duration"].isna().sum())

df.loc[misplaced, ["show_id", "title", "rating", "duration"]]

Rows moved: 0
Missing rating: 0
Missing duration: 0


,show_id,title,rating,duration


**3.2 Split `duration` into a number and a unit.** Movies say `90 min` and TV shows say `2 Seasons`, so one number column would mix minutes with seasons. We keep the number and the unit separately.

In [9]:
duration_parts = df["duration"].str.extract(
    r"(?i)^\s*(\d+)\s*(min|seasons?)\s*$"
)

df["duration_value"] = pd.to_numeric(duration_parts[0]).astype("Int64")
df["duration_unit"] = duration_parts[1].str.lower().replace({"seasons": "season"})

df.groupby("type")["duration_unit"].value_counts()

type     duration_unit
Movie    min              6131
TV Show  season           2676
Name: count, dtype: int64

In [10]:
df[["title", "type", "duration", "duration_value", "duration_unit"]].head(10)

,title,type,duration,duration_value,duration_unit
0,Dick Johnson Is Dead,Movie,90 min,90,min
1,Blood & Water,TV Show,2 Seasons,2,season
2,Ganglands,TV Show,1 Season,1,season
3,Jailbirds New Orleans,TV Show,1 Season,1,season
4,Kota Factory,TV Show,2 Seasons,2,season
5,Midnight Mass,TV Show,1 Season,1,season
6,My Little Pony: A New Generation,Movie,91 min,91,min
7,Sankofa,Movie,125 min,125,min
8,The Great British Baking Show,TV Show,9 Seasons,9,season
9,The Starling,Movie,104 min,104,min


**3.3 Filling `director`, `cast` and `country` with `Unknown`.** Many titles genuinely have no director or no listed country, and `Unknown` keeps those rows while staying distinguishable from real values.

In [11]:
text_columns = ["director", "cast", "country"]

for column in text_columns:
    df[column] = df[column].fillna("Unknown")

df[text_columns].isna().sum()

director    0
cast        0
country     0
dtype: int64

**3.4 Filling the remaining 7 missing ratings.** We can't verify the real rating from this file, so we label them `Not Rated` instead of guessing.

In [12]:
df["rating"] = df["rating"].fillna("Not Rated")

df["rating"].isna().sum()

np.int64(0)

**3.5 Converting `date_added` to a real date.** The text has leading spaces in 88 values, so we strip them first and give the exact format. Titles with no date stay as `NaT` (Pandas' missing date) because inventing a date would be misleading.

In [13]:
df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y")

print(df["date_added"].dtype)
print("Missing dates:", df["date_added"].isna().sum())
df["date_added"].agg(["min", "max"])

datetime64[us]
Missing dates: 10


min   2008-01-01
max   2021-09-25
Name: date_added, dtype: datetime64[us]

## Part 4 — Validate and export

**4.1 Proving the data is clean.** Each `assert` stops the notebook with an error if something is wrong.

In [14]:
assert df.shape == (8807, 14)
assert df.duplicated().sum() == 0
assert df["duration_value"].isna().sum() == 0
assert not df["rating"].str.contains("min", na=False).any()
assert df["date_added"].dtype.kind == "M"

print("All checks passed")
df.isna().sum()[lambda s: s > 0]

All checks passed


date_added    10
dtype: int64

**4.2 Before and after.** 

In [15]:
before = raw.isna().sum()
after = df.reindex(columns=raw.columns).isna().sum()

summary = pd.DataFrame({"missing_before": before, "missing_after": after})
summary[(summary > 0).any(axis=1)]

,missing_before,missing_after
director,2634,0
cast,825,0
country,831,0
date_added,10,10
rating,4,0
duration,3,0


**4.3 Export the cleaned data.** `index=False` stops Pandas writing an extra numbering column.

In [16]:
df.to_csv("netflix_titles_clean.csv", index=False)

**4.4 Reload the exported file to confirm it saved correctly.**

In [17]:
check = pd.read_csv("netflix_titles_clean.csv", parse_dates=["date_added"])
print(check.shape)
check.dtypes

(8807, 14)


show_id                      str
type                         str
title                        str
director                     str
cast                         str
country                      str
date_added        datetime64[us]
release_year               int64
rating                       str
duration                     str
listed_in                    str
description                  str
duration_value             int64
duration_unit                str
dtype: object

## Summary of decisions

- **Misplaced values:** 3 runtimes moved from `rating` to `duration`.
- **Duration:** split into `duration_value` and `duration_unit`, so minutes and seasons stay separate.
- **director / cast / country:** filled with `Unknown`, because many titles legitimately lack them.
- **rating:** 7 unverifiable values labelled `Not Rated`.
- **date_added:** parsed to datetime, with 10 genuine gaps left missing.
- **Result:** 8,807 rows kept, no duplicates, exported to `netflix_titles_clean.csv`.